torch.distributed 提供的集合通信（Collective Communication）算子，主要用于在进程组（Process Group）内的所有进程之间进行高效的数据同步与交换。这些算子是构建分布式训练（如数据并行、模型并行）的基石。
以下是 PyTorch 中最核心的集合通信算子及其详细解析：

## 🔄 核心集合通信算子
### all_reduce（全部归约）

- 作用：将所有进程中的张量进行指定的数学运算（如求和、求最大值等），并将最终结果广播到所有进程。
- 常用场景：
  - 数据并行（DDP）中的梯度同步。每个 GPU 算出局部梯度后，通过 all_reduce 求和并平均，确保所有 GPU 的模型参数更新一致。
- 代码示例：

假设我们有 4 个进程（rank 0, 1, 2, 3），我们将它们划分为两个组：
- Group A：包含 rank 0 和 1
- Group B：包含 rank 2 和 3

<font color='red'>全员参与建组：</font>
- 再次强调，即使 rank 2 和 3 不属于 group_a，它们也必须跟着执行 dist.new_group([0, 1])，否则 rank 0 和 1 在创建 group_a 时就会陷入死锁。

In [ ]:
import torch
import torch.distributed as dist
import os

# 假设已经通过 init_process_group 完成了全局初始化
# rank = int(os.getenv('RANK')) 

rank = dist.get_rank()
world_size = dist.get_world_size()

# 1. 全员同步创建子通信组
# 所有进程（0, 1, 2, 3）都必须按顺序执行这两行 new_group 代码
group_a = dist.new_group([0, 1])
group_b = dist.new_group([2, 3])

# 2. 各进程认领属于自己的组
my_group = None
if rank in [0, 1]:
    my_group = group_a
elif rank in [2, 3]:
    my_group = group_b

# 3. 在各自的子组内进行 all_reduce 通信
# 准备一个张量，值为当前 rank 的编号（转为 float 类型）
tensor = torch.tensor([float(rank)], dtype=torch.float32)

if my_group is not None:
    # 核心：传入 group=my_group，通信只会在该子组内发生
    # op=dist.ReduceOp.SUM 表示对组内所有进程的 tensor 进行求和
    dist.all_reduce(tensor, op=dist.ReduceOp.SUM, group=my_group)
    
    print(f"Rank {rank} 在子组内 all_reduce 后的结果: {tensor.item()}")

### all_gather（全部收集）
- 作用：收集进程组内所有进程的张量，并将它们拼接起来，最终每个进程都获得一份完全相同的、包含所有数据的完整张量。
- 常用场景：在计算全局的评估指标（如全局 Loss、准确率）时，需要将各卡上的局部数据汇总到所有卡上。
- 代码示例：

假设我们有 4 个进程（rank 0, 1, 2, 3），我们将它们划分为两个子组：
- Group A：包含 rank 0 和 1
- Group B：包含 rank 2 和 3
在这个示例中，all_gather 操作只会在各自的子组内收集数据，而不会跨组收集。

In [ ]:
import torch
import torch.distributed as dist

# 假设已经通过 init_process_group 完成了全局初始化
rank = dist.get_rank()
world_size = dist.get_world_size()

# 1. 全员同步创建子通信组（所有进程必须按相同顺序执行）
group_a = dist.new_group([0, 1])
group_b = dist.new_group([2, 3])

# 2. 各进程认领属于自己的组，并确定当前组的大小
my_group = None
group_size = 0

if rank in [0, 1]:
    my_group = group_a
    group_size = 2  # Group A 有 2 个进程
elif rank in [2, 3]:
    my_group = group_b
    group_size = 2  # Group B 有 2 个进程

# 3. 在各自的子组内进行 all_gather 通信
# 准备一个张量，值为当前 rank 的编号乘以 10（方便观察区分）
tensor = torch.tensor([rank * 10], dtype=torch.float32)

# ⚠️ 核心：需要提前分配好接收数据的空间
# 列表长度等于当前子组的进程数量（group_size），而不是全局的 world_size
gathered_list = [torch.zeros(1, dtype=torch.float32) for _ in range(group_size)]

if my_group is not None:
    # 传入 group=my_group，数据只会在该子组内互相收集
    dist.all_gather(gathered_list, tensor, group=my_group)
    
    print(f"Rank {rank} 在子组内 all_gather 后的结果: {[t.item() for t in gathered_list]}")

### broadcast（广播）
- 作用：将指定的源进程（src）的张量数据，<font color='red'>原封不动地复制并发送给组内所有其他进程。</font>
- 常用场景：
  - <font color='red'>在主节点（rank 0）加载好模型权重或超参数后，将其广播给其他所有工作节点，保证大家从同一个初始状态开始训练。</font>
- 代码示例：

在分布式训练中，使用 dist.broadcast 来同步主节点（rank 0）的模型权重或超参数，是确保所有工作节点从完全相同的初始状态开始训练的“标准动作”。

- 下面是一个完整的代码示例，模拟了 4 个进程，其中 rank 0 加载预训练权重，然后通过广播让其他所有进程同步这份权重：

In [ ]:
import torch
import torch.nn as nn
import torch.distributed as dist
import os

# 定义一个简单的模型
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(10, 5)
    
    def forward(self, x):
        return self.linear(x)

# 假设已经通过 init_process_group 完成了全局初始化
rank = dist.get_rank()
world_size = dist.get_world_size()

# 1. 实例化模型
model = SimpleModel()

# 2. 只有 rank 0 负责加载预训练权重或特定的超参数
if rank == 0:
    print(f"Rank {rank} 正在加载预训练权重...")
    # 模拟加载权重（实际使用时替换为你的权重文件路径）
    # pretrained_weights = torch.load("your_model_weights.pth", map_location="cpu")
    # model.load_state_dict(pretrained_weights)
    
    # 这里为了演示，我们手动修改 rank 0 的权重，代表“加载了特定状态”
    with torch.no_grad():
        model.linear.weight.fill_(1.0) 
        model.linear.bias.fill_(0.0)
    print(f"Rank {rank} 加载完成，当前权重均值: {model.linear.weight.mean().item():.4f}")

else:
    # 其他 rank 初始化时是随机权重
    print(f"Rank {rank} 初始随机权重均值: {model.linear.weight.mean().item():.4f}")

# 3. 【核心步骤】将 rank 0 的模型参数广播给所有其他进程
print(f"Rank {rank} 准备开始广播同步参数...")
# 遍历模型的所有参数，将 rank 0 的数据广播给所有人
for param in model.parameters():
    # src=0 表示数据源是 rank 0
    dist.broadcast(param.data, src=0)

# 4. 广播完成后，检查所有进程的权重是否已经一致
print(f"Rank {rank} 广播同步完成，当前权重均值: {model.linear.weight.mean().item():.4f}")

# 5. 接下来就可以将同步好参数的模型包装成 DDP 模型进行训练了
# ddp_model = nn.parallel.DistributedDataParallel(model, device_ids=[rank])

### reduce（归约）
- 作用：<font color='red'>将所有进程中的张量进行指定的数学运算，但最终的运算结果只保存在指定的目标进程（dst）中，其他进程的数据不改变（或被视为临时参与计算）。</font>
- 常用场景：
  - <font color='red'>只需要在主节点上汇总全局 Loss 进行打印或日志记录，而不需要其他节点保留汇总后的结果。</font>
- 代码示例：

使用 dist.reduce 可以将所有 GPU 计算出的局部 Loss 汇总到主节点（rank 0）进行统一的日志记录或监控，而其他节点不需要保留汇总后的结果。

In [ ]:
import torch
import torch.distributed as dist
import os

# 假设已经通过 init_process_group 完成了全局初始化
rank = dist.get_rank()
world_size = dist.get_world_size()

# 1. 模拟每个 GPU 进程独立计算出的局部 Loss
# 为了演示，这里让每个进程的 loss 等于它的 rank 值
local_loss = torch.tensor([float(rank)], dtype=torch.float32)
print(f"Rank {rank} 计算出的局部 Loss: {local_loss.item()}")

# 2. 【核心步骤】使用 dist.reduce 将全局 Loss 汇总到主节点 (rank 0)
# op=dist.ReduceOp.SUM 表示对所有进程的 local_loss 进行求和
# dst=0 表示最终的求和结果只保存在 rank 0 的 local_loss 变量中
dist.reduce(local_loss, dst=0, op=dist.ReduceOp.SUM)

# 3. 只有主节点 (rank 0) 进行最终的均值计算和日志打印
if rank == 0:
    # 汇总后，rank 0 的 local_loss 是所有进程 loss 的总和
    global_avg_loss = local_loss.item() / world_size
    print(f"\n[主节点 Rank 0] 汇总后的全局平均 Loss: {global_avg_loss:.4f}")
    # 在这里可以将 global_avg_loss 写入 TensorBoard、WandB 或日志文件
else:
    # 其他节点的 local_loss 在 reduce 操作后保持原样（或者被底层通信库修改为无意义的临时值）
    # 它们不需要关心汇总结果，继续执行后续任务即可
    print(f"Rank {rank} 不需要汇总结果，继续执行后续任务...")

### scatter（分散）
- 作用：gather 的逆操作。<font color='red'>将一个源进程（src）中的大张量列表，按顺序切分并分发给组内所有进程（包括源进程自己）。</font>
- 常用场景：<font color='green'>主节点将一整批数据或模型的不同部分，均匀分发给各个工作节点进行独立计算。</font>
- 代码示例：

In [ ]:
output = torch.zeros(2)
if dist.get_rank() == 0:
    # 只有 rank 0 准备好要分发的数据列表
    scatter_list = [torch.ones(2) * i for i in range(dist.get_world_size())]
else:
    scatter_list = None
# 分发后，rank 0 拿到 scatter_list[0]，rank 1 拿到 scatter_list[1]...
dist.scatter(output, scatter_list, src=0)

### gather（收集）
- 作用：scatter 的逆操作。<font color='red'>将所有进程各自的张量，全部收集并拼接到指定的目标进程（dst）中。</font>
- 常用场景：<font color='green'>各工作节点计算完局部结果后，统一汇报给主节点进行汇总处理。</font>
- 代码示例：

In [ ]:
gather_list = [torch.zeros(2) for _ in range(dist.get_world_size())] if dist.get_rank() == 0 else None
# 所有进程的 tensor 都会被收集到 rank 0 的 gather_list 中
dist.gather(tensor, gather_list, dst=0)

### reduce_scatter（归约分散）
- 作用：all_reduce 的一种高效变体。它先将各进程的张量分块，对各块分别进行归约（如求和），然后将第 i 块的结果分散到第 i 个进程。<font color='red'>每个进程最终只拿到全局归约结果的一部分。</font>
- 常用场景：在大模型优化器（如 ZeRO）或特定的集合通信算法（如 Ring All-Reduce 的第二阶段）中，用于减少单卡的显存占用和通信冗余。
- 代码示例：

  -  模拟了 4 个进程，每个进程拥有 4 个数据，最终通过 reduce_scatter 让每个进程只拿到全局求和结果的 1/4：

In [ ]:
import torch
import torch.distributed as dist

# 假设已经通过 init_process_group 完成了全局初始化
rank = dist.get_rank()
world_size = dist.get_world_size()

# 1. 准备数据
# 假设每个进程都有一个包含 4 个元素的张量，初始值为 rank * 10
# 例如：Rank 0 是 [0, 0, 0, 0], Rank 1 是 [10, 10, 10, 10] ...
local_tensor = torch.ones(4, dtype=torch.float32) * rank * 10
print(f"Rank {rank} 的初始局部数据: {local_tensor.tolist()}")

# 2. 准备接收缓冲区
# reduce_scatter 要求输出张量的大小必须是总数据量的 1/world_size
# 因为全局求和后的结果会被均分成 world_size 份，每个进程只拿 1 份
output_tensor = torch.zeros(1, dtype=torch.float32)

# 3. 【核心步骤】执行 reduce_scatter
# 输入需要将 local_tensor 拆分成 world_size 个块（这里拆成 4 个块，每块 1 个元素）
# op=SUM 表示先对所有进程的对应块进行求和
dist.reduce_scatter(output_tensor, list(local_tensor.chunk(world_size)), op=dist.ReduceOp.SUM)

# 4. 查看每个进程最终拿到的结果
print(f"Rank {rank} 归约分散后拿到的结果: {output_tensor.item()}")

### all_to_all（全交换）
- 作用：每个进程将自己的数据切分成 N 份，并将第 i 份发送给第 i 个进程，同时从所有进程接收属于自己那一份的数据。相当于一个全局的数据转置或混洗。
- 常用场景：在混合并行（如 MoE 模型、3D 并行中的流水线与张量并行切换）中，需要重新排列数据分布时使用。
- 代码示例：

In [ ]:
output = torch.zeros_like(input_tensor)
# 每个进程把自己的数据切分，并与其他进程交换对应的分块
dist.all_to_all(output, input_tensor)

## 📌 总结
- 这些算子在实际应用中，通常会配合你之前了解的 dist.new_group 使用。
- 通过指定 group 参数，可以让这些集合通信操作只在特定的子组（如张量并行组或流水线并行组）内发生，从而实现极其复杂且高效的 3D 并行训练架构。